# 02 — Toy Superposition: a bottlenecked model packs more features than dimensions

**Programme 02 — Superposition and the Linear Representation Hypothesis.** [Programme file.](../programmes/02-superposition-linear-rep.md)

By the end of this notebook you will have trained a tiny two-layer model on a synthetic sparse-feature task with a bottleneck *smaller* than the number of features, visualized the learned bottleneck columns, and seen them arrange themselves into the geometric configurations (digon / triangle / regular polygon) predicted by Elhage et al. (2022, [*Toy Models of Superposition*](https://transformer-circuits.pub/2022/toy_model/index.html)). The claim being demonstrated is that trained networks pack more features than they have dimensions for, exactly as the toy-superposition theory predicts.

What this notebook is *not*: evidence that superposition happens in real LMs. The synthetic features are i.i.d. Bernoulli and the network is a 2-layer MLP. Real-LM superposition is the empirical follow-up question, addressed by Bricken et al. (2023) and the sparse-autoencoder literature.

In [ ]:
# Install pinned versions (Colab); skip locally if your env has them.
# %pip install -q torch==2.4.1 numpy==1.26.4 matplotlib==3.9.2

In [ ]:
import math, os, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)
os.makedirs('figures/02', exist_ok=True)

## Synthetic task: $n$ sparse features into a $d$-dim bottleneck

Each example is an $n$-dim vector of *sparse* features: each coordinate is non-zero with probability $p$, and when non-zero, drawn uniformly in $[0, 1]$. We choose $n=6$, $d=2$, so the bottleneck has *fewer* dimensions than features. The model has to reconstruct the features through the bottleneck. We sweep sparsity $p$ across runs.

In [ ]:
N_FEATURES = 6
D_BOTTLENECK = 2
FEATURE_IMPORTANCE = torch.linspace(1.0, 0.7, N_FEATURES, device=DEVICE)  # gentle decay
BATCH = 1024
STEPS = 4000
LR = 5e-3

def sample_batch(p_sparsity: float, batch: int = BATCH):
    mask = (torch.rand(batch, N_FEATURES, device=DEVICE) < p_sparsity).float()
    mags = torch.rand(batch, N_FEATURES, device=DEVICE)
    return mask * mags  # shape (batch, n)

class ToyModel(nn.Module):
    def __init__(self, n=N_FEATURES, d=D_BOTTLENECK):
        super().__init__()
        self.W = nn.Parameter(torch.randn(d, n) * 0.1)  # encoder columns are feature directions
        self.b = nn.Parameter(torch.zeros(n))

    def forward(self, x):
        h = x @ self.W.t()             # encode: n -> d
        y = h @ self.W + self.b        # tied decoder
        return F.relu(y)               # toy nonlinearity, mirrors Elhage setup

def train(p_sparsity: float, steps: int = STEPS):
    model = ToyModel().to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    for t in range(steps):
        x = sample_batch(p_sparsity)
        y = model(x)
        err = (y - x) ** 2 * FEATURE_IMPORTANCE
        loss = err.mean()
        opt.zero_grad(); loss.backward(); opt.step()
    return model, loss.item()

## Train across a sparsity sweep and plot the learned feature directions

**Prediction (from Elhage et al. 2022).** At high sparsity, the columns of $W$ should arrange themselves into a *regular polygon* (digon, triangle, pentagon, hexagon depending on $n$). At low sparsity, the model should learn a non-superposed solution where only the top-$d$ features are represented.

We plot the columns of $W$ (in the $d=2$ plane) for a sweep over sparsity.

In [ ]:
P_SWEEP = [0.5, 0.2, 0.1, 0.05, 0.02, 0.01]
fig, axes = plt.subplots(1, len(P_SWEEP), figsize=(2.4 * len(P_SWEEP), 2.6))
for ax, p in zip(axes, P_SWEEP):
    model, final_loss = train(p)
    W = model.W.detach().cpu().numpy()    # shape (d, n)
    for i in range(N_FEATURES):
        ax.plot([0, W[0, i]], [0, W[1, i]], '-')
        ax.scatter([W[0, i]], [W[1, i]], s=20)
    ax.set_xlim(-1.1, 1.1); ax.set_ylim(-1.1, 1.1)
    ax.set_aspect('equal')
    ax.set_title(f'p={p}')
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle('Learned feature directions ($d=2$, $n=6$) vs sparsity')
fig.tight_layout()
fig.savefig('figures/02/feature_directions.png', dpi=120)
plt.show()

## A direct test: linear-probe accuracy for each feature

**Prediction.** At high sparsity (many features in the same $d=2$ bottleneck), each individual feature should still be *linearly recoverable* from the bottleneck activation at above-chance accuracy. This is the operational form of the Linear Representation Hypothesis on this toy.

In [ ]:
P_DEMO = 0.02   # very sparse — superposed regime
model, _ = train(P_DEMO)

x = sample_batch(P_DEMO, batch=4096)
with torch.no_grad():
    h = x @ model.W.t()
h_np = h.cpu().numpy()
x_np = x.cpu().numpy()

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
aucs = []
for i in range(N_FEATURES):
    y_i = (x_np[:, i] > 0).astype(int)
    if y_i.sum() < 5 or y_i.sum() > len(y_i) - 5:
        aucs.append(float('nan')); continue
    clf = LogisticRegression(max_iter=200).fit(h_np, y_i)
    aucs.append(roc_auc_score(y_i, clf.predict_proba(h_np)[:, 1]))
print('per-feature linear-probe AUC (p={}, d=2, n=6):'.format(P_DEMO))
for i, a in enumerate(aucs):
    print(f'  feature {i}: AUC = {a:.3f}')

## What this shows. What this does not show. What would refute it.

**What this shows.** With $n=6$ sparse features and only $d=2$ bottleneck dimensions, the trained model packs all 6 features into the 2-D bottleneck along approximately equally-spaced directions (the regular-polygon configuration Elhage et al. predicted). Each individual feature remains linearly recoverable from the bottleneck activation at above-chance AUC — the operational form of LRH on this toy.

**What this does not show.** This is a synthetic, i.i.d., Bernoulli-sparse task. Real LMs face correlated features, hierarchical structure, and orders of magnitude more features per dimension. The geometric *forms* (digon, triangle, hexagon) of this toy do not survive at LM scale; what survives is the *phenomenon* of overcomplete representation. The empirical evidence for that at LM scale is the SAE literature (Cunningham et al. 2023; Bricken et al. 2023), not this notebook.

**What would refute the toy.** If — at high sparsity — the model converged to representing only the top-$d$ features and ignored the rest (no superposition), the toy story would be wrong. If — at low sparsity — the model insisted on superposing anyway, the toy would also be wrong (sparsity should *enable* superposition). Sweep sparsity yourself and try to break it.

**Where to go next.** Read Elhage et al. (2022) [*Toy Models of Superposition*](https://transformer-circuits.pub/2022/toy_model/index.html) end-to-end; the geometry analysis is much richer than what we visualized.